In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.window import Window

In [0]:
%run /Workspace/Data_Ingestion_Atlikon/utilities

In [0]:
print(bronze_schema)
print(silver_schema)
print(gold_schema)

bronze
silver
gold


In [0]:
dbutils.widgets.text('catalog','fmcg','Catalog')
dbutils.widgets.text('data_source','gross_price','Data Source')

catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

base_path = f"s3://childcompanydata-myfirstproject/{data_source}/*.csv"
print(base_path)

s3://childcompanydata-myfirstproject/gross_price/*.csv


In [0]:
df = spark.read.format("csv")\
    .option("header", True)\
    .option("inferschema", True)\
    .load(base_path)\
    .withColumn("read_timestamp", F.current_timestamp())\
    .select("*", "_metadata.file_size", "_metadata.file_name")

display(df.limit(5))

product_id,month,gross_price,read_timestamp,file_size,file_name
25891101,2025/07/01,-84,2025-12-16T15:53:12.809Z,2741,gross_price.csv
25891101,01/08/2025,unknown,2025-12-16T15:53:12.809Z,2741,gross_price.csv
25891101,2025/09/01,84,2025-12-16T15:53:12.809Z,2741,gross_price.csv
25891101,2025-10-01,83,2025-12-16T15:53:12.809Z,2741,gross_price.csv
25891101,2025-11-01,83,2025-12-16T15:53:12.809Z,2741,gross_price.csv


In [0]:
df.write.format("delta")\
        .mode("overwrite")\
        .option("delta.enableChangeDataFeed", True)\
        .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

In [0]:
df_bronze = spark.sql(f"select * from {catalog}.{bronze_schema}.{data_source};")
df_bronze.show(5)

+----------+----------+-----------+--------------------+---------+---------------+
|product_id|     month|gross_price|      read_timestamp|file_size|      file_name|
+----------+----------+-----------+--------------------+---------+---------------+
|  25891101|2025/07/01|        -84|2025-12-16 15:59:...|     2741|gross_price.csv|
|  25891101|01/08/2025|    unknown|2025-12-16 15:59:...|     2741|gross_price.csv|
|  25891101|2025/09/01|         84|2025-12-16 15:59:...|     2741|gross_price.csv|
|  25891101|2025-10-01|         83|2025-12-16 15:59:...|     2741|gross_price.csv|
|  25891101|2025-11-01|         83|2025-12-16 15:59:...|     2741|gross_price.csv|
+----------+----------+-----------+--------------------+---------+---------------+
only showing top 5 rows


# Transformation

- Gross price to be converted to +ve or 0 for non-numerical data
- Normalize month column
- Bring product_code from products table

In [0]:
df_bronze.select("gross_price").distinct().show()

+-------------+
|  gross_price|
+-------------+
|          -84|
|      unknown|
|           84|
|           83|
|          -83|
|           68|
|           69|
|           85|
|not_available|
|           86|
|          108|
|           94|
|          100|
|          132|
|          138|
|          456|
|          493|
|         -493|
|          275|
|         -275|
+-------------+
only showing top 20 rows


In [0]:
df_bronze.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- month: string (nullable = true)
 |-- gross_price: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = true)
 |-- file_size: long (nullable = true)
 |-- file_name: string (nullable = true)



In [0]:
df_silver = df_bronze.withColumn(
  "gross_price",
  F.when(F.col("gross_price").rlike(r'^-?\d+(\.\d+)?$'),
         F.when(F.col("gross_price").cast("double") > 0, F.col("gross_price"))
          .otherwise(-1 * F.col("gross_price").cast("double")))
   .otherwise(0)
)

In [0]:
df_silver.select("gross_price").distinct().show()

+-----------+
|gross_price|
+-----------+
|       84.0|
|        0.0|
|       83.0|
|       68.0|
|       69.0|
|       85.0|
|       86.0|
|      108.0|
|       94.0|
|      100.0|
|      132.0|
|      138.0|
|      456.0|
|      493.0|
|      275.0|
|      300.0|
|      369.0|
|      432.0|
|       74.0|
|       79.0|
+-----------+
only showing top 20 rows


In [0]:
df_silver.select("month").distinct().show()

+----------+
|     month|
+----------+
|2025/07/01|
|01/08/2025|
|2025/09/01|
|2025-10-01|
|2025-11-01|
|2025-12-01|
|2025-07-01|
|2025-08-01|
|2025-09-01|
|2025/11/01|
|2025/08/01|
|01-09-2025|
|2025/10/01|
|01/12/2025|
|01/09/2025|
|01-12-2025|
|01-08-2025|
|01/10/2025|
+----------+



In [0]:
current_date_formats = ["yyyy-MM-dd", "yyyy/MM/dd", "dd/MM/yyyy", "dd-MM-yyyy"]

df_silver = df_silver.withColumn(
    "month",
    F.coalesce(
        F.try_to_date(F.col("month"), "yyyy-MM-dd"),
        F.try_to_date(F.col("month"), "yyyy/MM/dd"),
        F.try_to_date(F.col("month"), "dd/MM/yyyy"),
        F.try_to_date(F.col("month"), "dd-MM-yyyy"),
    )
)

In [0]:
df_silver.select("month").distinct().show()

+----------+
|     month|
+----------+
|2025-07-01|
|2025-08-01|
|2025-09-01|
|2025-10-01|
|2025-11-01|
|2025-12-01|
+----------+



In [0]:
df_products = spark.sql(f"select * from {catalog}.{silver_schema}.products;")
display(df_products.limit(5))

product_code,division,category,product,variant,product_id,read_timestamp,file_name,file_size
fe5a8036be4b9a787b7c0ae013fc752a8cfb6c55a2f7b2fd152a6380925e9c49,Dairy & Recovery,Recovery Dairy,SportsBar Greek Yogurt Pro Vanilla (120g),120g,25891402,2025-12-15T17:07:43.833Z,products.csv,1388
451f7167b28a25bde73995910e31c07dfa26411f1db47847f19e16747effbdaa,Hydration & Electrolytes,Electrolyte Mix,SportsBar Electrolyte Mix Lemon-Lime (5 Sachets),5 Sachets,25891603,2025-12-15T17:07:43.833Z,products.csv,1388
e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,Nutrition Bars,Energy Bars,SportsBar Energy Bar Choco Fudge (40g),40g,25891102,2025-12-15T17:07:43.833Z,products.csv,1388
3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,Breakfast Foods,Granola & Cereals,SportsBar Granola Crunch Honey Almond (400g),400g,25891301,2025-12-15T17:07:43.833Z,products.csv,1388
77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9,Dairy & Recovery,Recovery Dairy,SportsBar Greek Yogurt Pro Vanilla (80g),80g,25891403,2025-12-15T17:07:43.833Z,products.csv,1388


In [0]:
df_silver = df_silver.join(df_products.select("product_id", "product_code"), on="product_id", how="left")
display(df_silver.limit(5))

product_id,month,gross_price,read_timestamp,file_size,file_name,product_code
25891101,2025-07-01,84.0,2025-12-16T15:59:35.634Z,2741,gross_price.csv,e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843
25891101,2025-08-01,0.0,2025-12-16T15:59:35.634Z,2741,gross_price.csv,e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843
25891101,2025-09-01,84.0,2025-12-16T15:59:35.634Z,2741,gross_price.csv,e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843
25891101,2025-10-01,83.0,2025-12-16T15:59:35.634Z,2741,gross_price.csv,e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843
25891101,2025-11-01,83.0,2025-12-16T15:59:35.634Z,2741,gross_price.csv,e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843


In [0]:
df_silver_final = df_silver.select("product_id", "product_code", "month", "gross_price", "read_timestamp", "file_name", "file_size")

In [0]:
df_silver_final.write.format("delta")\
    .mode("overwrite")\
    .option("delta.enableChangeDataFeed", "true")\
    .option("mergeSchema", "true")\
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

# Gold Layer formation

In [0]:
df_silver = spark.sql(f"select * from {catalog}.{silver_schema}.{data_source}")
df_gold = df_silver.select("product_code", "month", "gross_price")

In [0]:
df_gold.write.format("delta")\
    .mode("overwrite")\
    .option("delta.enableChangeDataFeed", "true")\
    .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")